In [1]:
import polars as pl

In [2]:
df = pl.read_parquet("../data/parquet/*.parquet")
df

white_elo,black_elo,result,moves,opening
i32,i32,i8,str,str
2071,2030,1,"""e2e4 e7e5 g1f3 d7d6 d2d4 e5d4 …","""Philidor Defense: Exchange Var…"
2098,2152,1,"""e2e4 e7e5 g1f3 g8f6 b1c3 b8c6 …","""Four Knights Game: Scotch Vari…"
2080,2180,-1,"""e2e4 e7e5 g1f3 g8f6 b1c3 f8b4 …","""Russian Game: Three Knights Ga…"
2072,2067,1,"""e2e4 c7c5 g1f3 d7d6 d2d4 c5d4 …","""Sicilian Defense: Najdorf Vari…"
2091,2001,-1,"""g2g4 d7d5 f1g2 c7c6 h2h3 e7e5 …","""Grob Opening: Keene Defense"""
…,…,…,…,…
2131,2133,-1,"""e2e4 c7c5 g1f3 e7e6 d2d4 c5d4 …","""Sicilian Defense: Kan Variatio…"
2171,2112,1,"""e2e4 g7g6 d2d4 f8g7 b1c3 a7a6 …","""Modern Defense: Standard Line"""
2072,2290,1,"""g1f3 g7g6 d2d4 f8g7 c2c4 b7b6 …","""Zukertort Opening: Kingside Fi…"


In [3]:
# opening is whatever is before :, before , and without any #2, #3, etc. suffixes
# normalize whitespace so e.g. 'Vienna Game' and 'Vienna Game ' dedupe correctly
openings = (
    df["opening"]
    .str.split(":")
    .list.get(0)
    .str.split(",")
    .list.get(0)
    .str.replace(r"#\d+", "")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .unique()
    .sort()
)

for opening in openings:
    print(opening)

Alekhine Defense
Amar Opening
Amazon Attack
Anderssen Opening
Anderssen's Opening
Australian Defense
Barnes Defense
Barnes Opening
Benko Gambit
Benko Gambit Accepted
Benko Gambit Declined
Benoni Defense
Bird Opening
Bishop's Opening
Blackmar-Diemer Gambit
Blackmar-Diemer Gambit Accepted
Blackmar-Diemer Gambit Declined
Blumenfeld Countergambit
Blumenfeld Countergambit Accepted
Bogo-Indian Defense
Borg Defense
Budapest Defense
Canard Opening
Caro-Kann Defense
Carr Defense
Catalan Opening
Center Game
Center Game Accepted
Clemenz Opening
Colle System
Czech Defense
Danish Gambit
Danish Gambit Accepted
Danish Gambit Declined
Duras Gambit
Dutch Defense
East Indian Defense
Elephant Gambit
English Defense
English Opening
English Orangutan
Englund Gambit
Englund Gambit Complex
Englund Gambit Complex Declined
Englund Gambit Declined
Four Knights
Four Knights Game
Franco-Benoni Defense
French Defense
Gedult's Opening
Giuoco Piano
Goldsmith Defense
Grob Opening
Gruenfeld Defense
Grünfeld Defense
Gu

In [4]:
moves = df[0]["moves"].to_list()[0]
print(moves)

e2e4 e7e5 g1f3 d7d6 d2d4 e5d4 f3d4 f8e7 f1c4 g8f6 b1c3 e8g8 e1g1 a7a6 a2a3 c7c5 d4e2 b8c6 f2f4 c8g4 d1e1 b7b5 c4d5 f6d5 c3d5 c6d4 e2d4 c5d4 f4f5 f7f6 c1f4 a8c8 e1d2 g4h5 a1c1 h5f7 d2d4 c8c4 d4d3 f7d5 d3d5 g8h8 c2c3 d8b6 g1h1 a6a5 c1d1 c4c6 d5e6 b5b4 e6e7 f8d8 c3b4 a5b4 a3b4 h7h6 b4b5 c6c8 d1d6 d8d6 f4d6 b6b5 f1e1 c8e8 e7c7 e8e4 e1c1 b5f5 h2h3 e4e2 b2b4 f5f2 c7c8 h8h7 c8g4 e2b2 d6c5 f2d2 c1d1 d2c2 d1e1 f6f5 g4g3


In [9]:
moves = df["moves"].str.split(" ")
moves

moves
list[str]
"[""e2e4"", ""e7e5"", … ""g4g3""]"
"[""e2e4"", ""e7e5"", … ""e5d6""]"
"[""e2e4"", ""e7e5"", … ""c2g2""]"
"[""e2e4"", ""c7c5"", … ""e1e4""]"
"[""g2g4"", ""d7d5"", … ""f2f1""]"
…
"[""e2e4"", ""c7c5"", … ""g5b5""]"
"[""e2e4"", ""g7g6"", … ""d3g6""]"
"[""g1f3"", ""g7g6"", … ""f7d7""]"


In [14]:
all_moves = moves.explode().unique().sort()  # just experimenting, they are not all possible moves
all_moves

moves
str
"""a1a2"""
"""a1a3"""
"""a1a4"""
"""a1a5"""
"""a1a6"""
…
"""h8h3"""
"""h8h4"""
"""h8h5"""


In [ ]:
with open("../data/all_uci_moves.txt", "r") as f:
    all_uci_moves = [line.strip() for line in f]


tokenizer = {move: idx for idx, move in enumerate(all_uci_moves, start=2)}
tokenizer["<SOS>"] = 0
tokenizer["<EOS>"] = 1

{'a1h8': 2,
 'a1a8': 3,
 'a1g7': 4,
 'a1a7': 5,
 'a1f6': 6,
 'a1a6': 7,
 'a1e5': 8,
 'a1a5': 9,
 'a1d4': 10,
 'a1a4': 11,
 'a1c3': 12,
 'a1a3': 13,
 'a1b2': 14,
 'a1a2': 15,
 'a1h1': 16,
 'a1g1': 17,
 'a1f1': 18,
 'a1e1': 19,
 'a1d1': 20,
 'a1c1': 21,
 'a1b1': 22,
 'a2g8': 23,
 'a2a8': 24,
 'a2f7': 25,
 'a2a7': 26,
 'a2e6': 27,
 'a2a6': 28,
 'a2d5': 29,
 'a2a5': 30,
 'a2c4': 31,
 'a2a4': 32,
 'a2b3': 33,
 'a2a3': 34,
 'a2h2': 35,
 'a2g2': 36,
 'a2f2': 37,
 'a2e2': 38,
 'a2d2': 39,
 'a2c2': 40,
 'a2b2': 41,
 'a2b1': 42,
 'a2a1': 43,
 'a3f8': 44,
 'a3a8': 45,
 'a3e7': 46,
 'a3a7': 47,
 'a3d6': 48,
 'a3a6': 49,
 'a3c5': 50,
 'a3a5': 51,
 'a3b4': 52,
 'a3a4': 53,
 'a3h3': 54,
 'a3g3': 55,
 'a3f3': 56,
 'a3e3': 57,
 'a3d3': 58,
 'a3c3': 59,
 'a3b3': 60,
 'a3b2': 61,
 'a3a2': 62,
 'a3c1': 63,
 'a3a1': 64,
 'a4e8': 65,
 'a4a8': 66,
 'a4d7': 67,
 'a4a7': 68,
 'a4c6': 69,
 'a4a6': 70,
 'a4b5': 71,
 'a4a5': 72,
 'a4h4': 73,
 'a4g4': 74,
 'a4f4': 75,
 'a4e4': 76,
 'a4d4': 77,
 'a4c4': 78,
 'a4b4'